#  Part 1 — Demo & Kiểm thử

Kiểm thử bốn hàm của **Part 1**:

| Hàm verify | Kiểm tra |
|---|---|
| `verify_solve` | Nghiệm hệ phương trình `Ax = b` (khử Gauss) |
| `verify_determinant` | Định thức `det(A)` |
| `verify_inverse` | Ma trận nghịch đảo `A⁻¹` |
| `verify_rank` | Hạng (rank) của ma trận |

Mỗi hàm verify đối chiếu kết quả tự cài đặt với **NumPy** làm chuẩn.  
Các file `gaussian.py`, `determinant.py`, `inverse.py`, `rank_basis.py` nằm **cùng thư mục** với notebook này.


## 0. Import

In [1]:
import numpy as np
import random

from gaussian import gaussian_eliminate
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

print("Import thành công!")


Import thành công!


## 1. Dữ liệu test dùng chung

Định nghĩa danh sách **10 test case chuẩn** và hàm sinh **ma trận ngẫu nhiên** cho test lớn.  
Tất cả các hàm verify bên dưới đều dùng chung bộ `gauss_cases` này.


In [2]:
def generate_large_test(n):
    A = [[random.randint(-5, 5) for _ in range(n)] for _ in range(n)]
    b = [random.randint(-10, 10) for _ in range(n)]
    return A, b

# 10 test case chuẩn
gauss_cases = [
    {"name": "He co nghiem duy nhat",
     "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], "b": [8, -11, -3]},
    {"name": "Pivot = 0",
     "A": [[0, 2, 1], [1, -2, -3], [-1, 1, 2]], "b": [-8, 0, 3]},
    {"name": "Vo so nghiem",
     "A": [[1, 1, 1], [2, 2, 2], [3, 3, 3]], "b": [3, 6, 9]},
    {"name": "Vo nghiem",
     "A": [[1, 1], [1, 1]], "b": [1, 2]},
    {"name": "Suy bien",
     "A": [[1, 2], [2, 4]], "b": [3, 6]},
    {"name": "Ma tran don vi",
     "A": [[1, 0, 0], [0, 1, 0], [0, 0, 1]], "b": [5, -3, 2]},
    {"name": "Zero matrix",
     "A": [[0, 0], [0, 0]], "b": [0, 0]},
    {"name": "Gan suy bien",
     "A": [[1, 1, 1], [1, 1.0000001, 1], [1, 1, 1.0000001]],
     "b": [3, 3.0000001, 3.0000001]},
    {"name": "4x4",
     "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]], "b": [10,8,9,11]},
    {"name": "Rank < n",
     "A": [[1,2,3],[2,4,6],[0,0,0]], "b": [6,12,0]},
]

# Test lớn ngẫu nhiên
for size in [30, 50, 80, 100, 200, 500, 1000]:
    A, b = generate_large_test(size)
    gauss_cases.append({"name": f"Large random {size}x{size}",
                         "A": A, "b": b, "large": True})

print(f"Tổng số test case: {len(gauss_cases)}")


Tổng số test case: 17


## 2. `verify_solve` — Kiểm thử giải hệ phương trình Ax = b

**Thuật toán:** Khử Gauss với partial pivoting → thế ngược.  
**Verify:** Tính lại `A @ x` và so sánh với `b` bằng `np.allclose`.  
- Nếu hệ **vô nghiệm** hoặc **vô số nghiệm** → hàm trả về chuỗi mô tả, không so sánh mảng.  
- Nếu hệ có **nghiệm duy nhất** → kiểm tra `A @ x ≈ b`.


In [3]:
def verify_solve(A, b, x_custom):
    if isinstance(x_custom, str) or (isinstance(x_custom, list) and isinstance(x_custom[0], str)):
        print("   >> Nghiệm x: [Hệ vô nghiệm/vô số nghiệm - Không so sánh mảng]")
        return True
    A_np, b_np, x_np = np.array(A, dtype=float), np.array(b, dtype=float), np.array(x_custom, dtype=float)
    return np.allclose(A_np @ x_np, b_np)

# ── Chạy test ──
print("=" * 60)
print("KIEM THU: verify_solve (giai he phuong trinh Ax = b)")
print("=" * 60)

for idx, tc in enumerate(gauss_cases):
    name     = tc["name"]
    A, b     = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    print(f"\n Test case {idx+1}: {name}")
    if not is_large:
        print(f"   A = {A}")
        print(f"   b = {b}")
    else:
        print(f"   (Ma tran lon {len(A)}x{len(A[0])} - khong hien thi)")
    try:
        _, my_x, _ = gaussian_eliminate(A, b)
        ok = verify_solve(A, b, my_x)
        print("   OK Passed" if ok else "   FAIL: Sai nghiem Ax = b")
    except Exception as e:
        print(f"   CRASH: {e}")


KIEM THU: verify_solve (giai he phuong trinh Ax = b)

 Test case 1: He co nghiem duy nhat
   A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
   b = [8, -11, -3]
   OK Passed

 Test case 2: Pivot = 0
   A = [[0, 2, 1], [1, -2, -3], [-1, 1, 2]]
   b = [-8, 0, 3]
   OK Passed

 Test case 3: Vo so nghiem
   A = [[1, 1, 1], [2, 2, 2], [3, 3, 3]]
   b = [3, 6, 9]
Không có pivot tại cột 1.
Không có pivot tại cột 2.
   >> Nghiệm x: [Hệ vô nghiệm/vô số nghiệm - Không so sánh mảng]
   OK Passed

 Test case 4: Vo nghiem
   A = [[1, 1], [1, 1]]
   b = [1, 2]
Không có pivot tại cột 1.
   >> Nghiệm x: [Hệ vô nghiệm/vô số nghiệm - Không so sánh mảng]
   OK Passed

 Test case 5: Suy bien
   A = [[1, 2], [2, 4]]
   b = [3, 6]
Không có pivot tại cột 1.
   >> Nghiệm x: [Hệ vô nghiệm/vô số nghiệm - Không so sánh mảng]
   OK Passed

 Test case 6: Ma tran don vi
   A = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
   b = [5, -3, 2]
   OK Passed

 Test case 7: Zero matrix
   A = [[0, 0], [0, 0]]
   b = [0, 0]
Không có pivot t

## 3. `verify_determinant` — Kiểm thử tính định thức

**Công thức:**  
$\det(A) = (-1)^{\text{swaps}} \times \prod_{i} U_{ii}$ 

Trong đó $U$ là ma trận tam giác trên sau khử Gauss, `swaps` là số lần hoán đổi dòng.

**Verify:** So sánh với `np.linalg.det(A)` bằng `np.isclose`.  
- Ma trận **suy biến** (det = 0): kết quả phải bằng 0.  
- Ma trận **đơn vị**: det = 1.


In [4]:
def verify_determinant(A, det_custom):
    A_np = np.array(A, dtype=float)
    return np.isclose(det_custom, np.linalg.det(A_np))

# ── Chạy test ──
print("=" * 60)
print("KIEM THU: verify_determinant (tinh dinh thuc)")
print("=" * 60)

for idx, tc in enumerate(gauss_cases):
    name     = tc["name"]
    A, b     = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    print(f"\n Test case {idx+1}: {name}")
    if not is_large:
        print(f"   A = {A}")
    else:
        print(f"   (Ma tran lon {len(A)}x{len(A[0])} - khong hien thi)")
    try:
        my_det = determinant(A)
        ok = verify_determinant(A, my_det)
        if not is_large:
            print(f"   det(A) = {my_det:.6f}")
        print("   OK Passed" if ok else "   FAIL: Sai dinh thuc")
    except Exception as e:
        print(f"   CRASH: {e}")


KIEM THU: verify_determinant (tinh dinh thuc)

 Test case 1: He co nghiem duy nhat
   A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
   det(A) = -1.000000
   OK Passed

 Test case 2: Pivot = 0
   A = [[0, 2, 1], [1, -2, -3], [-1, 1, 2]]
   det(A) = 1.000000
   OK Passed

 Test case 3: Vo so nghiem
   A = [[1, 1, 1], [2, 2, 2], [3, 3, 3]]
Không có pivot tại cột 1.
Không có pivot tại cột 2.
   det(A) = -0.000000
   OK Passed

 Test case 4: Vo nghiem
   A = [[1, 1], [1, 1]]
Không có pivot tại cột 1.
   det(A) = 0.000000
   OK Passed

 Test case 5: Suy bien
   A = [[1, 2], [2, 4]]
Không có pivot tại cột 1.
   det(A) = -0.000000
   OK Passed

 Test case 6: Ma tran don vi
   A = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
   det(A) = 1.000000
   OK Passed

 Test case 7: Zero matrix
   A = [[0, 0], [0, 0]]
Không có pivot tại cột 0.
Không có pivot tại cột 1.
   det(A) = 0.000000
   OK Passed

 Test case 8: Gan suy bien
   A = [[1, 1, 1], [1, 1.0000001, 1], [1, 1, 1.0000001]]
   det(A) = 0.000000
   OK Passed

/Users/lengoctuyen/Documents/Allmyprojects/Matrix-Decomposition/.venv/lib/python3.14/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: overflow encountered in det
  r = _umath_linalg.det(a, signature=signature)


   OK Passed

 Test case 17: Large random 1000x1000
   (Ma tran lon 1000x1000 - khong hien thi)
   OK Passed


## 4. `verify_inverse` — Kiểm thử ma trận nghịch đảo

**Thuật toán:** Giải $A\mathbf{x}_i = \mathbf{e}_i$ cho từng vector đơn vị $\mathbf{e}_i$,  
rồi ghép các nghiệm lại thành các cột của $A^{-1}$.

**Verify:** Kiểm tra $A \cdot A^{-1} \approx I$ bằng `np.allclose`.  
- Ma trận **không khả nghịch** (det = 0) → trả về chuỗi thông báo, bỏ qua so sánh.


In [ ]:
def verify_inverse(A, inv_custom):
    if isinstance(inv_custom, str):
        print("   >> Nghich dao: [Ma tran suy bien - Khong so sanh mang]")
        return True
    A_np, inv_np = np.array(A, dtype=float), np.array(inv_custom, dtype=float)
    return np.allclose(A_np @ inv_np, np.eye(A_np.shape[0]))

# ── Chạy test ──
print("=" * 60)
print("KIEM THU: verify_inverse (tinh ma tran nghich dao)")
print("=" * 60)

for idx, tc in enumerate(gauss_cases):
    name     = tc["name"]
    A, b     = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    print(f"\n Test case {idx+1}: {name}")
    if not is_large:
        print(f"   A = {A}")
    else:
        print(f"   (Ma tran lon {len(A)}x{len(A[0])} - khong hien thi)")
    try:
        my_inv = inverse(A)
        ok = verify_inverse(A, my_inv)
        print("   OK Passed" if ok else "   FAIL: Sai nghich dao")
    except Exception as e:
        print(f"   CRASH: {e}")


KIEM THU: verify_inverse (tinh ma tran nghich dao)

 Test case 1: He co nghiem duy nhat
   A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
   OK Passed

 Test case 2: Pivot = 0
   A = [[0, 2, 1], [1, -2, -3], [-1, 1, 2]]
   OK Passed

 Test case 3: Vo so nghiem
   A = [[1, 1, 1], [2, 2, 2], [3, 3, 3]]
Không có pivot tại cột 1.
Không có pivot tại cột 2.
   >> Nghich dao: [Ma tran suy bien - Khong so sanh mang]
   OK Passed

 Test case 4: Vo nghiem
   A = [[1, 1], [1, 1]]
Không có pivot tại cột 1.
   >> Nghich dao: [Ma tran suy bien - Khong so sanh mang]
   OK Passed

 Test case 5: Suy bien
   A = [[1, 2], [2, 4]]
Không có pivot tại cột 1.
   >> Nghich dao: [Ma tran suy bien - Khong so sanh mang]
   OK Passed

 Test case 6: Ma tran don vi
   A = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
   OK Passed

 Test case 7: Zero matrix
   A = [[0, 0], [0, 0]]
Không có pivot tại cột 0.
Không có pivot tại cột 1.
   >> Nghich dao: [Ma tran suy bien - Khong so sanh mang]
   OK Passed

 Test case 8: Gan suy bien
   

## 5. `verify_rank` — Kiểm thử hạng ma trận

**Thuật toán:** Gauss-Jordan với partial pivoting → đưa về dạng RREF,  
đếm số hàng khác 0 = số pivot = rank.

**Verify:** So sánh với `np.linalg.matrix_rank(A)`.  
- Ma trận **zero**: rank = 0.  
- Ma trận **đơn vị** n×n: rank = n.  
- Hệ có **vô số nghiệm** (dòng phụ thuộc): rank < n.


In [ ]:
def verify_rank(A, rank_custom):
    A_np = np.array(A, dtype=float)
    return rank_custom == np.linalg.matrix_rank(A_np)

# ── Chạy test ──
print("=" * 60)
print("KIEM THU: verify_rank (tinh hang ma tran)")
print("=" * 60)

for idx, tc in enumerate(gauss_cases):
    name     = tc["name"]
    A, b     = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    print(f"\n Test case {idx+1}: {name}")
    if not is_large:
        print(f"   A = {A}")
    else:
        print(f"   (Ma tran lon {len(A)}x{len(A[0])} - khong hien thi)")
    try:
        my_rank, _ = rank_and_basis(A)
        ok = verify_rank(A, my_rank)
        if not is_large:
            print(f"   rank(A) = {my_rank}")
        print("   OK Passed" if ok else "   FAIL: Sai rank")
    except Exception as e:
        print(f"   CRASH: {e}")


## 6. Tổng kết — Chạy toàn bộ Part 1

Chạy cả 4 verify cùng lúc trên toàn bộ test case và in bảng kết quả tổng hợp.


In [ ]:
print("=" * 60)
print("TONG KET PART 1")
print("=" * 60)

passed = 0
for idx, tc in enumerate(gauss_cases):
    name     = tc["name"]
    A, b     = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    print(f"\n Test case {idx+1}: {name}")
    if not is_large:
        print(f"   A = {A}")
        print(f"   b = {b}")
    else:
        print(f"   (Ma tran lon {len(A)}x{len(A[0])} - khong hien thi)")
    try:
        my_det  = determinant(A)
        my_rank = rank_and_basis(A)[0]
        _, my_x, _ = gaussian_eliminate(A, b)
        my_inv  = inverse(A)
        errors = []
        if not verify_solve(A, b, my_x):     errors.append("Sai nghiem Ax = b")
        if not verify_determinant(A, my_det): errors.append("Sai dinh thuc")
        if not verify_inverse(A, my_inv):     errors.append("Sai nghich dao")
        if not verify_rank(A, my_rank):       errors.append("Sai rank")
        if not errors:
            passed += 1
        print("   OK Passed" if not errors else "   FAIL: " + ", ".join(errors))
    except Exception as e:
        print(f"   CRASH: {e}")

print("\n" + "=" * 60)
print(f"KET QUA: {passed}/{len(gauss_cases)} test case PASSED")
print("=" * 60)
